# Approximating the reverse causal cone with a Tree Tensor Network

For a depth-$p$ QAOA circuit, $\langle Z_iZ_j\rangle$ for an edge $(i,j)$ only depends on its **reverse causal cone**: the sub-circuit built from graph edges within distance $p$ of $i$ or $j$ (this is what `LightConeEvaluator` already exploits, via exact statevector simulation of the reduced circuit).

**Key fact**: if the graph's girth is large enough relative to $p$, this causal-cone subgraph is itself a *tree*. It can then be represented directly as a Tree Tensor Network (TTN) whose topology matches the causal cone -- every QAOA gate acts on tree-adjacent qubits, so no SWAP routing is needed. Bounding the TTN bond dimension $\chi$ gives a controlled approximation of $\langle Z_iZ_j\rangle$.

We use [pytreenet](https://github.com/Udhaya2/pytreenet)'s low-level TTN primitives (`contract_nodes` + `absorb_into_open_legs` + `split_node_svd`), the same building blocks pytreenet's own TEBD implementation uses to apply a two-site gate and truncate back down.

In [15]:
import networkx as nx
import numpy as np
import pytreenet as ptn
from scipy.linalg import expm
import time

from qaoa_training_pipeline.evaluation.light_cone import LightConeEvaluator
from qaoa_training_pipeline.evaluation.mps_evaluator import MPSEvaluator
from qaoa_training_pipeline.evaluation.efficient_depth_one import EfficientDepthOneEvaluator
from qaoa_training_pipeline.utils.graph_utils import graph_to_operator, operator_to_graph

X, _, Z = ptn.pauli_matrices()

## TTN causal-cone evaluator

`causal_cone_energy` builds the causal-cone tree of a single edge (reusing `LightConeEvaluator.make_radius_edges`), represents it as a `|+>`-product TTN, applies the QAOA layers as tree-local RZZ/RX gates with bond dimension `max_bond_dim`, and reads off `<Z_iZ_j>`. `ttn_causal_cone_evaluate` sums this over all edges to get the QAOA energy.

In [2]:
def causal_cone_energy(graph, edge, betas, gammas, max_bond_dim):
    """<Z_i Z_j> for `edge`, approximated via a bond-dimension-limited TTN of its causal cone."""
    lce = LightConeEvaluator()
    lce.graph = graph
    depth = len(betas)
    cone_edges = lce.make_radius_edges(edge, radius=depth)

    tree = nx.Graph()
    tree.add_edges_from((u, v, {"weight": w}) for (u, v), w in cone_edges.items())
    if not tree.has_edge(*edge):
        tree.add_edge(*edge, weight=graph[edge[0]][edge[1]].get("weight", 1.0))
    if not nx.is_tree(tree):
        # Cone has a cycle (graph girth too small for this depth): fall back to
        # LightConeEvaluator's exact statevector simulation for just this edge.
        circ, obs = lce.make_radius_circuit(edge, list(betas) + list(gammas))
        circ = circ.decompose()
        result = lce.primitive.run([(circ, obs)]).result()
        return float(np.real(result[0].data.evs))

    # Build a |+>-product TTN whose topology is exactly the causal-cone tree.
    root = edge[0]
    degree = dict(tree.degree)
    plus_tensor = lambda n_neighbours: np.full((1,) * n_neighbours + (2,), 1 / np.sqrt(2), dtype=complex)

    ttn = ptn.TreeTensorNetworkState()
    ttn.add_root(ptn.Node(identifier=str(root)), plus_tensor(degree[root]))
    next_free_slot = {root: 0}  # next unused neighbour-leg on each node already in the TTN
    for parent, child in nx.bfs_edges(tree, root):
        ttn.add_child_to_parent(
            ptn.Node(identifier=str(child)), plus_tensor(degree[child]),
            0, str(parent), next_free_slot[parent],
        )
        next_free_slot[parent] += 1
        next_free_slot[child] = 1  # leg 0 of the child is now taken by its parent

    svd_params = ptn.SVDParameters(max_bond_dim=max_bond_dim, rel_tol=0.0, total_tol=1e-14)

    def apply_two_site_gate(id1, id2, gate):
        # Same 4-step pattern pytreenet's TEBD uses to apply+truncate a nearest-neighbour gate.
        u_legs, v_legs = ttn.legs_before_combination(id1, id2)
        ttn.contract_nodes(id1, id2, new_identifier="contr")
        ttn.absorb_into_open_legs("contr", gate)
        ttn.split_node_svd("contr", u_legs, v_legs, u_identifier=id1, v_identifier=id2, svd_params=svd_params)

    for beta, gamma in zip(betas, gammas):
        for u, v, w in tree.edges(data="weight"):
            rzz = expm(-1j * gamma * w * np.kron(Z, Z)).reshape(2, 2, 2, 2)
            apply_two_site_gate(str(u), str(v), rzz)
        for node in tree.nodes:
            ttn.absorb_into_open_legs(str(node), expm(-1j * beta * X))

    zz = ttn.operator_expectation_value(ptn.TensorProduct({str(edge[0]): Z, str(edge[1]): Z}))
    return float(np.real(zz / ttn.scalar_product()))  # renormalize away any truncation-induced norm drift


def ttn_causal_cone_evaluate(cost_op, params, max_bond_dim):
    """QAOA energy as a sum of TTN-approximated causal-cone edge expectation values."""
    depth = len(params) // 2
    betas, gammas = params[:depth], params[depth:]
    graph = operator_to_graph(cost_op)
    return sum(
        w * causal_cone_energy(graph, (u, v), betas, gammas, max_bond_dim)
        for u, v, w in graph.edges(data="weight")
    )

## Validation against exact statevector simulation

We use the McGee graph (girth 7, cubic, 24 nodes), so that every edge's depth-2 causal cone is guaranteed to be a tree.

In [ ]:
# mcgee_graph = nx.LCF_graph(24, [12, 7, -7], 8)
mcgee_graph = nx.random_regular_graph(5, 100)
cost_op = graph_to_operator(mcgee_graph, pre_factor=-0.5)  # standard MaxCut convention

rng = np.random.default_rng(0)
depth = 1
params = rng.uniform(-1, 1, size=2 * depth)  # [beta_0, ..., beta_{p-1}, gamma_0, ..., gamma_{p-1}]

# exact_energy = StatevectorEvaluator().evaluate(cost_op, list(params))
t_mps = time.time()
mps_energy = MPSEvaluator(bond_dim_circuit=32, use_swap_strategy=True).evaluate(cost_op, params)
t_ttn = time.time()
ttn_energy = ttn_causal_cone_evaluate(cost_op, params, max_bond_dim=32)
t_eff_depth_1 = time.time()
eff_depth_one = EfficientDepthOneEvaluator().evaluate(cost_op, params)
# print(f"exact statevector energy:  {exact_energy:.10f}")
print(f"TTN causal-cone energy:    {ttn_energy:.10f}")
print(f"MPS evaluator energy:    {mps_energy:.10f}")
print(f"Efficient depth one evaluator energy:    {eff_depth_one:.10f}")

print(f"TTN causal-cone energy:    {time.time() - t_ttn}")
print(f"MPS evaluator energy:    {time.time() - t_mps}")
print(f"Efficient depth one evaluator energy:    {time.time() - t_eff_depth_1}")

TTN causal-cone energy:    -32.2282839770
MPS evaluator energy:    -10.9993080210
Efficient depth one evaluator energy:    -32.2282839770


## Effect of the bond dimension

At fixed depth, the causal cone's entanglement is bounded, so the TTN energy converges to the exact value once `max_bond_dim` is large enough -- this is the accuracy/cost knob for larger, deeper causal cones where exact simulation is not an option.

In [ ]:
import matplotlib.pyplot as plt

# EfficientDepthOneEvaluator is analytically exact at p = 1, so it stands in for the
# statevector reference (which does not scale to the graph sizes used here).
exact_energy = EfficientDepthOneEvaluator().evaluate(cost_op, list(params))

bond_dimensions = [1, 2, 3, 4, 6, 8, 16, 32]
errors = [abs(ttn_causal_cone_evaluate(cost_op, params, chi) - exact_energy) for chi in bond_dimensions]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(bond_dimensions, np.maximum(errors, 1e-16), "o-")
ax.set_xlabel(r"TTN bond dimension $\chi$")
ax.set_ylabel(r"$|E_{\mathrm{TTN}} - E_{\mathrm{exact}}|$")
ax.set_yscale("log")
fig.tight_layout()

## Benchmark against the other evaluators

We now compare the TTN causal-cone evaluator against `MPSEvaluator`, `EfficientDepthOneEvaluator` and `LightConeEvaluator` on random regular graphs with a guaranteed large girth. Two constraints fix the experimental setup.

**Depth.** `EfficientDepthOneEvaluator` only exists at $p=1$, so the comparison runs at $p=1$. There it is *analytically exact*, which makes it the natural accuracy reference for the other three evaluators. Its own curve therefore sits on the clipping floor of the error panel by construction.

**Girth.** The depth-$p$ causal cone of an edge $(i,j)$ is the induced subgraph on $B_p(i)\cup B_p(j)$, so it is a tree exactly when the graph has no cycle short enough to fit inside that ball -- that is, when the girth is at least $2p+3$. For $p=1$ this means girth $\geq 5$. `networkx` ships `random_regular_graph`, but it gives no control over the girth: a random cubic graph almost always contains triangles and squares, which would send every edge down `causal_cone_energy`'s non-tree fallback and quietly turn the "TTN" evaluator into `LightConeEvaluator`. So we add a girth-constrained generator below.

In [ ]:
def random_regular_graph_min_girth(degree, num_nodes, min_girth, seed=None, max_restarts=500):
    """A random `degree`-regular graph on `num_nodes` nodes with girth at least `min_girth`.

    Built with the pairing model: half-edges ("stubs") are matched at random, but a pairing
    is rejected whenever it would close a cycle shorter than `min_girth`. A run that paints
    itself into a corner is restarted from scratch.
    """
    rng = np.random.default_rng(seed)
    for _ in range(max_restarts):
        graph = _attempt_min_girth_pairing(degree, num_nodes, min_girth, rng)
        if graph is not None:
            return graph

    raise RuntimeError(
        f"No {degree}-regular graph on {num_nodes} nodes with girth >= {min_girth} "
        f"found in {max_restarts} restarts."
    )


def _attempt_min_girth_pairing(degree, num_nodes, min_girth, rng, stall_limit=200):
    """One pairing-model attempt. Returns None if it gets stuck."""
    graph = nx.empty_graph(num_nodes)
    stubs = list(np.repeat(np.arange(num_nodes), degree))
    rng.shuffle(stubs)
    stalls = 0

    while stubs:
        idx1, idx2 = rng.choice(len(stubs), size=2, replace=False)
        node1, node2 = int(stubs[idx1]), int(stubs[idx2])

        # Adding (node1, node2) closes a cycle of length d(node1, node2) + 1, so the two
        # endpoints must already be at least `min_girth - 1` hops apart.
        too_close = nx.single_source_shortest_path_length(graph, node1, cutoff=min_girth - 2)
        if node1 == node2 or node2 in too_close:
            stalls += 1
            if stalls > stall_limit:
                return None
            continue

        graph.add_edge(node1, node2, weight=1.0)
        for idx in sorted((idx1, idx2), reverse=True):
            stubs.pop(idx)
        stalls = 0

    return graph


def all_cones_are_trees(graph, depth):
    """True if every edge's depth-`depth` reverse causal cone is a tree."""
    lce = LightConeEvaluator()
    lce.graph = graph
    return all(
        nx.is_tree(nx.Graph(list(lce.make_radius_edges(edge, radius=depth))))
        for edge in graph.edges
    )


DEGREE = 3                  # degree of the random regular graphs
DEPTH = 1                   # QAOA depth; EfficientDepthOneEvaluator only exists at p = 1
MIN_GIRTH = 2 * DEPTH + 3   # girth needed for every depth-p causal cone to be a tree

demo_graph = random_regular_graph_min_girth(DEGREE, 40, MIN_GIRTH, seed=0)
print(f"degree {DEGREE}, girth {nx.girth(demo_graph)} (need >= {MIN_GIRTH}), "
      f"every depth-{DEPTH} cone is a tree: {all_cones_are_trees(demo_graph, DEPTH)}")

In [ ]:
GRAPH_SIZES = [10, 14, 20, 40, 60, 80]  # number of nodes; `DEGREE * n` must be even
NUM_INSTANCES = 3                       # random instances per size
BOND_DIM = 32                           # shared by the TTN and the MPS evaluator
BENCH_PARAMS = [0.4, 0.6]               # [beta_0, gamma_0], near the p=1 cubic-MaxCut optimum

# Evaluators are rebuilt for every measurement, so each timing includes its own setup cost
# and nothing is cached across instances.
BENCH_EVALUATORS = {
    "TTN causal cone": lambda op, prm: ttn_causal_cone_evaluate(op, prm, max_bond_dim=BOND_DIM),
    "MPS": lambda op, prm: MPSEvaluator(
        bond_dim_circuit=BOND_DIM, use_swap_strategy=True
    ).evaluate(op, prm),
    "EfficientDepthOne": lambda op, prm: EfficientDepthOneEvaluator().evaluate(op, prm),
    "LightCone": lambda op, prm: LightConeEvaluator().evaluate(op, prm),
}

records = []
for num_nodes in GRAPH_SIZES:
    for instance in range(NUM_INSTANCES):
        bench_graph = random_regular_graph_min_girth(
            DEGREE, num_nodes, MIN_GIRTH, seed=1000 * num_nodes + instance
        )
        bench_op = graph_to_operator(bench_graph, pre_factor=-0.5)  # MaxCut convention

        energies, runtimes = {}, {}
        for name, evaluate in BENCH_EVALUATORS.items():
            tic = time.perf_counter()
            energies[name] = float(evaluate(bench_op, BENCH_PARAMS))
            runtimes[name] = time.perf_counter() - tic

        # EfficientDepthOne is analytically exact at p = 1, so it is the accuracy reference.
        reference = energies["EfficientDepthOne"]
        records += [
            {
                "evaluator": name,
                "num_nodes": num_nodes,
                "instance": instance,
                "runtime": runtimes[name],
                "rel_error": abs(energies[name] - reference) / abs(reference),
            }
            for name in BENCH_EVALUATORS
        ]

        print(f"n={num_nodes:3d} instance={instance}  " + "  ".join(
            f"{name}: {runtimes[name]:7.2f}s" for name in BENCH_EVALUATORS), flush=True)

In [ ]:
import matplotlib.pyplot as plt

STYLES = {
    "TTN causal cone": {"color": "tab:red", "marker": "o"},
    "MPS": {"color": "tab:blue", "marker": "s"},
    "EfficientDepthOne": {"color": "tab:green", "marker": "^"},
    "LightCone": {"color": "tab:orange", "marker": "D"},
}
FLOOR = 1e-16  # relative errors are clipped here so the exact evaluators stay on the log axis

fig, (ax_time, ax_error) = plt.subplots(2, 1, sharex=True, figsize=(6.4, 7.2))

for name, style in STYLES.items():
    rows = [r for r in records if r["evaluator"] == name]
    sizes = sorted({r["num_nodes"] for r in rows})
    runtimes = [[r["runtime"] for r in rows if r["num_nodes"] == n] for n in sizes]
    errors = [[max(r["rel_error"], FLOOR) for r in rows if r["num_nodes"] == n] for n in sizes]

    for axis, per_size in ((ax_time, runtimes), (ax_error, errors)):
        means = [np.mean(values) for values in per_size]
        low = [mean - np.min(values) for mean, values in zip(means, per_size)]
        high = [np.max(values) - mean for mean, values in zip(means, per_size)]
        axis.errorbar(sizes, means, yerr=[low, high], label=name, capsize=3, lw=1.6, ms=5,
                      **style)
        # Individual instances, so the instance-to-instance spread stays visible.
        axis.scatter([n for n, values in zip(sizes, per_size) for _ in values],
                     [value for values in per_size for value in values],
                     s=9, alpha=0.35, zorder=1, color=style["color"])

ax_time.set_ylabel("execution time [s]")
ax_time.set_yscale("log")
ax_time.set_title(f"{DEGREE}-regular, girth $\\geq$ {MIN_GIRTH}, $p={DEPTH}$, "
                  f"$\\chi={BOND_DIM}$, {NUM_INSTANCES} instances/size")
ax_time.legend(fontsize=9, loc="upper left")

ax_error.set_ylabel(r"$|E - E_\mathrm{exact}| / |E_\mathrm{exact}|$")
ax_error.set_xlabel("number of nodes $n$")
ax_error.set_yscale("log")
ax_error.set_ylim(bottom=FLOOR / 50)  # room under the floor line for its label
ax_error.axhline(FLOOR, color="gray", ls=":", lw=1)
ax_error.text(0.99, 0.015, "exact, clipped at machine precision", fontsize=7, color="gray",
              transform=ax_error.transAxes, ha="right", va="bottom")

for axis in (ax_time, ax_error):
    axis.grid(alpha=0.3, which="both")

fig.tight_layout()

### Reading the plot

The two cone-based evaluators, `LightConeEvaluator` and the TTN, are exact to machine precision at every size: their cost per edge is set by the causal cone, which for a degree-$d$ graph at depth $p$ has a size independent of $n$, so both scale linearly in the number of edges. The TTN is consistently the cheaper of the two -- roughly $4\times$ here -- because a cone that is a tree is contracted directly, whereas `LightConeEvaluator` builds, transpiles and statevector-simulates a Qiskit circuit per edge.

`MPSEvaluator` behaves very differently. At $n=10$ the bond dimension $\chi=32$ already saturates the maximum Schmidt rank $2^{n/2}$, so it is exact; beyond that it truncates, and because a random regular graph admits no one-dimensional ordering that keeps its cuts small, the relative error climbs steeply and does not level off -- about $2\%$ at $n=14$, $16\%$ at $n=20$, and $60\%$ by $n=80$. Its runtime grows steeply too, since the line swap strategy needs $O(n)$ layers of $O(n)$ gates, and the instance-to-instance spread at fixed $n$ is large: the bond dimension an instance actually reaches depends on how well the swap network happens to suit it.

`EfficientDepthOneEvaluator` is the fastest by an order of magnitude and is exact, but only because $p=1$ is precisely the case its closed-form density-matrix recursion covers. It is the right tool inside this plot and unavailable outside it, which is what makes the cone-based evaluators worth their extra cost: they carry over unchanged to $p\geq 2$, where the TTN's bond dimension becomes a real accuracy knob rather than a formality.

Two caveats when re-running this. The TTN error sits at the floor because a depth-1 cone on a cubic graph is only 6 qubits, far below $\chi=32$; to make the approximation actually bite, either lower `BOND_DIM`, or raise `DEPTH` -- `MIN_GIRTH` follows automatically, but you then have to drop `EfficientDepthOne` from `BENCH_EVALUATORS` and use `LightConeEvaluator` as the reference. And `MPS` dominates the total runtime of the benchmark cell (a few minutes for the settings above), so shrink `GRAPH_SIZES` or `NUM_INSTANCES` for a quicker pass.